In [ ]:
import pickle
from sklearn.linear_model import LinearRegression
import numpy as np

Below is a dummy ML model built from linear regression:

In [ ]:
X = np.random.rand(3,3)
y = np.random.rand(3,2)
print(X, y)

In [ ]:
model = LinearRegression()
model.fit(X,y)
model.predict(np.array([1,2,3]).reshape(1,-1))[0]

In [ ]:
picklefile = open('trained_model', 'wb')
#pickle the object and store it in a file
pickle.dump(model, picklefile)

In [ ]:
#check that the object is correctly pickled and works when unpickled
del model
picklefile = open('trained_model', 'rb')
new_model = pickle.load(picklefile)
new_model.predict(np.array([1,2,3]).reshape(1,-1))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
train_pricing_decisions = pd.read_csv('train_prices_decisions_2025.csv')
test_user_info = pd.read_csv('test_user_info_2025.csv')

In [ ]:
train_pricing_decisions = pd.read_csv('../../data/train_prices_decisions_2025.csv')
test_user_info = pd.read_csv('../../data/test_user_info_2025.csv')

In [ ]:
train_pricing_decisions.head()

In [ ]:
print(f"Dataset shape: {train_pricing_decisions.shape}")
print(f"Number of customers: {len(train_pricing_decisions)}")
print(f"Column names: {list(train_pricing_decisions.columns)}")
print(f"Data types:\n{train_pricing_decisions.dtypes}")
print(f"Missing values:\n{train_pricing_decisions.isnull().sum()}")
print("First 5 rows:")
print(train_pricing_decisions.head())

In [ ]:
# import data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

In [ ]:
# From HW 3, finding if individual covariates are higher than median
train_pricing_decisions['Covariate1_high'] = (

    train_pricing_decisions['Covariate1'] >= train_pricing_decisions['Covariate1'].median()
).astype(int)

train_pricing_decisions['Covariate2_high'] = (
    train_pricing_decisions['Covariate2'] >= train_pricing_decisions['Covariate2'].median()
).astype(int)

train_pricing_decisions['Covariate3_high'] = (
    train_pricing_decisions['Covariate3'] >= train_pricing_decisions['Covariate3'].median()
).astype(int)


**Feature engineering: Binary covariate groups**

Following the HW3 approach, we create binary features by splitting each covariate at its median. This allows us to capture different customer segments (high vs low covariate values) which respond differently to prices, as discovered in our EDA.

In [ ]:
# Building feature matrix
def prepare_features(df):
    X = pd.DataFrame({
        "price": df["price_item"].values,
        "Cov1_high": df["Covariate1_high"].values,
        "Cov2_high": df["Covariate2_high"].values,
        "Cov3_high": df["Covariate3_high"].values})
    X["price_x_Cov1"] = X["price"] * X["Cov1_high"]
    X["price_x_Cov2"] = X["price"] * X["Cov2_high"]
    X["price_x_Cov3"] = X["price"] * X["Cov3_high"]
    return X

X_train = prepare_features(train_pricing_decisions)
y_train = train_pricing_decisions["item_bought"].astype(int)

print("Feature matrix shape: " + str(X_train.shape))
print("Percentage that bought the item: " + str(round(y_train.mean(), 4)))

**Build feature matrix with interactions**

The `prepare_features()` function creates our feature matrix with:
- Base features: price and the 3 binary covariate indicators
- Interaction terms: price × Cov1, price × Cov2, price × Cov3

These interactions capture the fact that price sensitivity varies by customer segment (e.g., high Cov1 customers may be less price-sensitive). This is the same approach as HW3 Problem 2.

In [ ]:
train_pricing_decisions

In [ ]:
# Split traing and test data, same from HW3
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
model = LogisticRegression(solver='liblinear',penalty='l2',C=1.0,fit_intercept=True,max_iter=5000,random_state=42)
model.fit(X_train_split, y_train_split)


print("Coefficients:")
print("Intercept: " + str(round(model.intercept_[0], 4)))
for feat, coef in zip(X_train.columns, model.coef_[0]):
    print (feat + ": " + str(round(coef, 4)))

**Train logistic regression model**

We split data 80/20 for training and validation, then train a logistic regression with:
- L2 regularization (penalty='l2', C=1.0) to prevent overfitting
- liblinear solver (works well for small-medium datasets)
- This predicts P(purchase=1 | price, covariates)

The coefficients show how each feature affects purchase probability. Negative coefficients for price interactions indicate that price sensitivity varies by segment.

In [ ]:
# Evaluate model
y_val_pred_proba = model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, y_val_pred_proba)
logloss = log_loss(y_val, y_val_pred_proba)

print("Validation AUC: " + str(round(auc_score, 4)))
print("Validation Log Loss: " + str(round(logloss, 4)))

**Model evaluation metrics**

We evaluate the model on validation data using:
- **AUC (Area Under ROC Curve)**: Measures how well the model discriminates between purchase vs no-purchase (higher is better, max=1.0)
- **Log Loss**: Measures the quality of predicted probabilities (lower is better, min=0.0)

Log loss is particularly important for pricing because we need accurate probability estimates to calculate expected revenue = price × P(purchase).

In [ ]:
with open('trained_model', 'wb') as f:
    pickle.dump(model, f)

print("✓ Model saved successfully to 'trained_model'")
print(f"Model type: {type(model).__name__}")
print(f"Number of features: {len(model.coef_[0])}")

### Save the Trained Model

We save the trained logistic regression model using pickle so it can be loaded by the static pricing CSV generator to compute optimal prices for all test customers and loaded by the dynamic pricing agent to make real-time pricing decisions during the head-to-head competition

The saved model contains:
- All fitted coefficients (intercept + 7 feature weights)
- The prediction method to compute P(purchase | price, covariates)
- The model configuration (solver, regularization, etc.)

In [ ]:
with open('trained_model', 'rb') as f:
    loaded_model = pickle.load(f)

print(f"Model type: {type(loaded_model).__name__}")
print(f"Model class: {loaded_model.__class__}")
print(f"Model parameters:")
print(f"Solver: {loaded_model.solver}")
print(f"Regularization (C): {loaded_model.C}")
print(f"Max iterations: {loaded_model.max_iter}")

print(f"Model coefficients shape: {loaded_model.coef_.shape}")
print(f"Number of features: {loaded_model.coef_.shape[1]}")

print(f"Intercept: {loaded_model.intercept_[0]:.4f}")
print("Feature coefficients:")
feature_names = ['price', 'Covariate1_high', 'Covariate2_high', 'Covariate3_high',
                 'price_x_Cov1', 'price_x_Cov2', 'price_x_Cov3']
for name, coef in zip(feature_names, loaded_model.coef_[0]):
    print(f"{name:18s}: {coef:8.4f}")

y_loaded_pred = loaded_model.predict_proba(X_val)[:, 1]

match = np.allclose(y_val_pred_proba, y_loaded_pred)
print(f"Loaded model predictions match original: {match}")

### Model Verification

We verify the saved model by:
1. **Loading it back** from the file using pickle
2. **Checking model properties** - type, parameters, coefficients
3. **Testing predictions** - make predictions on validation set and compare with original model

This ensures:
- The model was saved correctly
- It can be loaded without errors
- It produces identical predictions to the original trained model

If predictions match, the model is ready for use in Phase 3 (CSV generation) and the dynamic pricing agent.